# Blue Owls Data Engineer Assessment — README

## Overview

This project implements a complete data pipeline using a medallion architecture (Bronze -> Silver -> Gold) on top of a simulated e-commerce API. The goal was not just to move data, but to design a system that is resilient, reproducible, and analytically reliable under imperfect real-world conditions.

The pipeline is built using Python for ingestion, PySpark for transformations and SparkSQL for Quering/validating Data, with outputs stored as CSVs.

---

## Technical Decisions & Reasoning

### 1. Medallion Architecture (Bronze → Silver → Gold)

I chose a medallion architecture because it cleanly separates concerns:

* **Bronze** preserves raw data exactly as received (append-only)
* **Silver** standardizes, validates, and deduplicates data
* **Gold** models the data for analytics (star schema)

This separation allowed me to:

* Debug issues without re-ingesting data
* Apply data quality rules without losing raw data
* Keep business logic isolated from ingestion logic

---

### 2. Python for Ingestion, PySpark for Transformation

I used:

* **Python (requests + pandas)** for API ingestion
* **PySpark** for transformations

Reasoning:

* Python is simpler and more flexible for handling API retries, authentication, and pagination
* PySpark is better suited for scalable transformations and joins

This split reflects how pipelines are often built in production (ingestion vs processing layers separated).

---

### 3. Deterministic Surrogate Keys (Hash-Based)

Instead of using sequential IDs, I used **SHA-256 hashes of natural keys** to generate surrogate keys.

Example:

* `customer_key = sha2(customer_unique_id)`
* `order_item_sk = sha2(order_id + order_item_id)`

**Why:**

* Guarantees reproducibility across runs
* Avoids dependency on execution order
* Works well in distributed environments

I considered using row_number(), but rejected it because it is not deterministic across runs.

---

### 4. Incremental Ingestion with Manifest Tracking

Rather than reloading all data every time, I implemented **incremental ingestion** using a manifest file.

The manifest tracks:

* Last successfully ingested date per endpoint

**Why this approach:**

* Avoids duplicate data in Bronze
* Makes the pipeline idempotent
* Reduces unnecessary API calls

I also added a safety buffer (re-fetching the previous day) to handle late-arriving data.

---

### 5. Silver Layer Upsert Strategy

Silver represents the **current state of data**, so I implemented upserts using:

* Window functions (`row_number`)
* Ordering by `_ingested_at`

This ensures:

* Latest record wins
* No duplicates in Silver
* Bronze remains append-only

---

### 6. Data Quality Strategy

Instead of dropping bad data, I:

* Flagged records using `_is_valid`
* Preserved all data for traceability

Examples:

* Orders where delivery date < purchase date -> invalid
* Negative prices -> invalid
* Missing IDs -> invalid

This avoids silent data loss and allows downstream debugging.

---

### 7. Referential Integrity in Gold Layer

To ensure no orphan keys:

* Fact table joins were carefully constructed
* Validation checks explicitly verify FK integrity

I also used deterministic keys so joins are always consistent.

---

## API Failure Handling & Resilience Strategy

The API intentionally simulates real-world failures, so I designed the ingestion layer to handle:

### 1. Token Expiry (401)

* Automatically refresh token and retry request

### 2. Server Errors (500) & Rate Limits (429)

* Implemented exponential backoff with jitter
* Retries up to a fixed limit

### 3. Pagination Failures

* Each page request is retried independently
* Prevents losing entire dataset due to one failure

### 4. Idempotency

* Manifest ensures already ingested data is not reprocessed
* Deduplication ensures safe re-runs

One key design choice here was **retrying only failed requests instead of restarting the entire ingestion**, which reduces unnecessary load and improves reliability.

---

## Assumptions & Trade-offs

### 1. Missing Values

* Product category -> replaced with `"unknown"`
* Dates -> allowed nulls but validated logically

### 2. Payment Allocation

* Payment distributed proportionally by item price
* Assumes price reflects contribution to payment

### 3. Late Delivery Logic

* `is_late_delivery = days_delivery_vs_estimate > 0`
* Null if delivery not completed

### 4. Trade-offs

**CSV vs Delta/Parquet**

* Used CSV to match assignment requirements
* Trade-off: no schema enforcement or ACID guarantees

**Performance vs Simplicity**

* Did not optimize joins (broadcast, partitioning)
* Focused on correctness and clarity

**Full vs Incremental Loads**

* Implemented incremental ingestion
* But Silver still recomputes full dataset (simpler, safer)

---

## Production Improvements (Azure / Microsoft Fabric)

If this were deployed in production, I would make the following changes:

### 1. Storage & Format

* Replace CSV with **Delta Lake on ADLS Gen2**
* Benefits:

  * ACID transactions
  * Schema enforcement
  * Time travel

---

### 2. Orchestration

* Use **Azure Data Factory**
* Schedule:

  * Daily ingestion jobs
  * Downstream transformation dependencies

---

### 3. Monitoring & Alerting

* Integrate with **Azure Monitor / Log Analytics**
* Track:

  * API failure rates
  * Data quality metrics
  * Pipeline duration

---

### 4. CI/CD

* Use **GitHub Actions / Azure DevOps**
* Automate:

  * Testing
  * Deployment
  * Notebook execution validation

---

### 5. Security

* Store credentials in **Azure Key Vault**
* Avoid hardcoding secrets
* Use managed identities where possible

---

### 6. Cost Optimisation

* Use **auto-scaling Spark clusters**
* Partition data by date
* Cache frequently used tables

---

### 7. Data Quality Framework

* Add automated checks (e.g., Great Expectations)
* Track:

  * Null rates
  * Duplicate rates
  * FK violations

---

## Final Thoughts

The main focus of this implementation was **building a resilient and explainable pipeline**, rather than just completing all features.

A key decision I made was prioritizing:

* Deterministic transformations
* Clear separation of layers
* Explicit validation

This ensures the pipeline is not only correct, but also maintainable and production-ready with minimal changes.

---
